### Objetivo del ejercicio
En este mini-lab vas a construir un pipeline ETL completo con PySpark, paso a paso. El objetivo es practicar cómo leer, transformar y guardar datos usando las funciones básicas que has aprendido.

Extraemos datos del csv ---> Transformamos --> Cargamos en parquet

In [94]:
# importamos librerias
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import column
from pyspark.sql.types import StructField, StringType, DoubleType, IntegerType, StructType

In [95]:
# creamos la sesion
spark = (((SparkSession.builder
         .appName("aplicacion")
         .master("local[*]"))
         .config("spark.driver.memory", "2g"))
         .getOrCreate())
print(f"version: {spark.version}")

version: 4.2.0


In [96]:
schema= StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("category", StringType(), True),
    StructField("date", StringType(), True),
    StructField("_reg_corrupt", StringType(), True)
])

# lectura dataset
df_csv = spark.read.csv("p2_transactions.csv", header=True, schema=schema, sep=",",
                        mode= "PERMISSIVE", columnNameOfCorruptRecord = "_reg_corrupt",
                        nullValue="NULL", emptyValue="")

df_csv.show(30)

+--------------+-----------+-------+------+-----------+----------+------------+
|transaction_id|customer_id|country|amount|   category|      date|_reg_corrupt|
+--------------+-----------+-------+------+-----------+----------+------------+
|             1|        101|  Spain| 120.5|Electronics|2024-01-15|        NULL|
|             2|        102| France|  75.0|    Fashion|2024-01-17|        NULL|
|             3|        103|Germany| 210.3|       Home|2024-01-20|        NULL|
|             4|        104|  Spain|  33.0|      Books|2024-02-02|        NULL|
|             5|        105| France| 450.0|Electronics|2024-02-05|        NULL|
|             6|        106|Germany|  89.9|    Fashion|2024-02-06|        NULL|
|             7|        107|  Italy|  25.0|      Books|2024-02-07|        NULL|
|             8|        108|  Spain| 310.0|       Home|2024-02-08|        NULL|
|             9|        109| France|  12.0|      Books|2024-02-09|        NULL|
|            10|        110|Germany| 199

In [97]:
df_csv.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- date: string (nullable = true)
 |-- _reg_corrupt: string (nullable = true)



In [98]:
# hacemos limpieza del dataset
from functools import reduce
from operator import or_
from pyspark.sql import functions as F

# detectamos filas con nulos
df_csv = df_csv.drop("_reg_corrupt")

listado_nulos = []# almacenaremos listado de valores nulos

for columna, tipo_dato in df_csv.dtypes:
    if tipo_dato == "string": # si la columna es un string
        # == CORREGIDO: Comparamos contra "" en lugar de " " ==
        valor = (F.col(columna) == "") | (F.col(columna) == "NULL") | (F.col(columna).isNull()) 
    elif tipo_dato in ("float", "double"): # examinamos los tipo decimales
        valor = (F.isnan(F.col(columna))) | (F.col(columna).isNull())
    else: # es de tipo entero, fechas o booleanos
        valor = (F.col(columna).isNull())
    listado_nulos.append(valor)
    
# unimos resultados
filas = reduce(or_, listado_nulos)
        
df_csv.filter(filas).show()

+--------------+-----------+-------+------+-----------+----------+
|transaction_id|customer_id|country|amount|   category|      date|
+--------------+-----------+-------+------+-----------+----------+
|            21|        121|   NULL|  95.0|    Fashion|2024-02-23|
|            22|        122|Germany|  NULL|Electronics|2024-02-24|
|            23|        123|  Spain|  45.0|       NULL|2024-02-25|
|            24|        124| France| 115.5|       Home|      NULL|
|            25|       NULL|  Italy|  60.0|      Books|2024-02-27|
|            26|        126|  Spain|  NULL|    Fashion|2024-02-28|
|            27|        127|   NULL| 320.0|Electronics|2024-03-01|
|            28|        128|Germany|  15.9|       NULL|2024-03-02|
|            29|        129| France|  NULL|       Home|2024-03-03|
|            30|        130|  Italy| 180.0|      Books|      NULL|
|            33|       NULL|   NULL|  NULL|       NULL|      NULL|
+--------------+-----------+-------+------+-----------+-------

In [99]:
# eliminamos valores nulos
df_csv = df_csv.dropna(how="any")
df_csv.count()

24

In [100]:
#filtramos transacciones > 100
df_csv = df_csv.filter(F.col("amount") > 100)
df_csv.count()

12

In [101]:
# guardamos en formato parquet -> pais
((df_csv.write.mode("overwrite") # sobreescribimos las carpetas si ya existen
    .partitionBy("country")) # partimos los datos por country
    .option("compression", "snappy") # forma de comprimir los datos
    .parquet("parquet_country") #ruta donde se almacenan
)

In [102]:
# comprobamos para leer por un pais
df_parquet = spark.read.parquet("parquet_country")

# hacemos un filtrado por francia
df_parquet_francia = df_parquet.filter(F.col("country") == "France")
df_parquet_francia.show()

+--------------+-----------+------+-----------+----------+-------+
|transaction_id|customer_id|amount|   category|      date|country|
+--------------+-----------+------+-----------+----------+-------+
|             5|        105| 450.0|Electronics|2024-02-05| France|
|            13|        113| 125.0|       Home|2024-02-13| France|
|            17|        117| 310.2|Electronics|2024-02-19| France|
+--------------+-----------+------+-----------+----------+-------+



## Parte opcional

In [103]:
# generar nueva columna
df_parquet = df_parquet.withColumn("amount_usd", F.col("amount") * 1.1)
df_parquet.show(5)

+--------------+-----------+------+-----------+----------+-------+------------------+
|transaction_id|customer_id|amount|   category|      date|country|        amount_usd|
+--------------+-----------+------+-----------+----------+-------+------------------+
|             3|        103| 210.3|       Home|2024-01-20|Germany|231.33000000000004|
|            10|        110| 199.0|Electronics|2024-02-10|Germany|             218.9|
|            32|        132| 410.2|Electronics|2024-03-06|Germany|            451.22|
|            12|        112| 480.5|Electronics|2024-02-12|  Italy| 528.5500000000001|
|            20|        120| 205.9|       Home|2024-02-22|  Italy|226.49000000000004|
+--------------+-----------+------+-----------+----------+-------+------------------+
only showing top 5 rows


In [104]:
# guardamos resultado en parquet particionado por año y mes

# modificamos el tipo de fecha
df_parquet = (df_parquet
              .withColumn("date_ts", F.to_timestamp(F.col("date")))
              .withColumn("year", F.year(F.col("date_ts")))
              .withColumn("month", F.month(F.col("date_ts")))
)
df_parquet.show(5)

+--------------+-----------+------+-----------+----------+-------+------------------+-------------------+----+-----+
|transaction_id|customer_id|amount|   category|      date|country|        amount_usd|            date_ts|year|month|
+--------------+-----------+------+-----------+----------+-------+------------------+-------------------+----+-----+
|             3|        103| 210.3|       Home|2024-01-20|Germany|231.33000000000004|2024-01-20 00:00:00|2024|    1|
|            10|        110| 199.0|Electronics|2024-02-10|Germany|             218.9|2024-02-10 00:00:00|2024|    2|
|            32|        132| 410.2|Electronics|2024-03-06|Germany|            451.22|2024-03-06 00:00:00|2024|    3|
|            12|        112| 480.5|Electronics|2024-02-12|  Italy| 528.5500000000001|2024-02-12 00:00:00|2024|    2|
|            20|        120| 205.9|       Home|2024-02-22|  Italy|226.49000000000004|2024-02-22 00:00:00|2024|    2|
+--------------+-----------+------+-----------+----------+------

In [105]:
# almacenamos por año y mes en parquet
df_parquet= (df_parquet.write.mode("overwrite")
             .partitionBy("year", "month")
             .option("compression", "snappy")
             .parquet("parquet_year_month"))

In [106]:
spark.stop()